# MIMM Patients Endpoint Baseline

This notebook builds a strong tabular baseline using **all modalities**, with **radiology aggregated by mean per patient**, **per-modality PCA**, and:

- `LogisticRegression` for `OS_3_label`, `OS_4_label`, `OS_5_label`, `OS_6_label`, `OS_9_label`, `OS_12_label`
- `Ridge` regression for continuous `os_months`

The evaluation uses nested cross-validation to avoid optimistic model selection.

## 1. Configuration

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path('/Users/marcalbesa/Desktop/TFM/data/MIMM')
RANDOM_STATE = 42
N_OUTER_SPLITS = 5
N_INNER_SPLITS = 4
RADIO_AGGREGATION_METHOD = 'mean'

CLASSIFICATION_TARGETS = ['OS_3_label', 'OS_4_label', 'OS_5_label', 'OS_6_label', 'OS_9_label', 'OS_12_label']
REGRESSION_TARGET = 'os_months'

PCA_VARIANCE_GRID = [0.90, 0.95, 0.99]
LOGREG_C_GRID = [0.01, 0.1, 1.0, 10.0]
LOGREG_CLASS_WEIGHT_GRID = [None, 'balanced']
RIDGE_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'RADIO_AGGREGATION_METHOD: {RADIO_AGGREGATION_METHOD}')

PROJECT_ROOT: /Users/marcalbesa/Desktop/TFM/git_exp/methods
DATA_ROOT: /Users/marcalbesa/Desktop/TFM/data/MIMM
RADIO_AGGREGATION_METHOD: mean


## 2. Imports And Data Loading

In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)


def collapse_patient_rows_mean(df, id_col='patient'):
    feature_cols = [c for c in df.columns if c != id_col]
    dense_df = df[[id_col] + feature_cols].copy()
    dense_df[feature_cols] = dense_df[feature_cols].apply(pd.to_numeric, errors='coerce')
    return dense_df.groupby(id_col, as_index=False, sort=False)[feature_cols].mean()


def ensure_single_row_per_patient(df, modality_name, id_col='patient', collapse_strategy='mean'):
    duplicated = df[id_col].value_counts()
    duplicated = duplicated[duplicated > 1]
    if duplicated.empty:
        return df

    preview = duplicated.index.tolist()[:10]
    print(
        f"[{modality_name}] duplicated patients detected: {len(duplicated)}. "
        f"Collapsing by {collapse_strategy}. Example IDs: {preview}"
    )
    if collapse_strategy != 'mean':
        raise ValueError(f"Unsupported collapse strategy: {collapse_strategy}")
    return collapse_patient_rows_mean(df, id_col=id_col)


def load_mimm_modalities_mean(data_root: Path):
    patients_df = pd.read_csv(data_root / 'patients_mimm.csv')
    required_label_cols = ['patient', 'os_months'] + CLASSIFICATION_TARGETS
    missing_label_cols = [c for c in required_label_cols if c not in patients_df.columns]
    if missing_label_cols:
        raise ValueError(
            f"patients_mimm.csv is missing required endpoint columns: {missing_label_cols}"
        )
    labels_df = patients_df[required_label_cols].copy()

    path_df = pd.read_csv(data_root / 'pathology_mimm.csv')
    path_df = path_df.rename(columns=lambda x: x.replace('embedding_', 'patho_'))
    keep = ['patient'] + [c for c in path_df.columns if c.startswith('patho_')]
    path_df = ensure_single_row_per_patient(path_df[keep], 'path')

    radio_df = pd.read_csv(data_root / 'radiology_mimm.csv')
    radio_df = radio_df.rename(columns=lambda x: x.replace('pred_', 'radio_'))
    radio_df = radio_df.drop(columns=['image_path', 'lesion_tag'], errors='ignore')
    keep = ['patient'] + [c for c in radio_df.columns if c.startswith('radio_')]
    radio_df = collapse_patient_rows_mean(radio_df[keep], id_col='patient')

    clin_df = pd.read_csv(data_root / 'clinical_mimm.csv')
    clin_df = clin_df.rename(columns=lambda x: f'clin_{x}' if x != 'patient' else x)
    clin_df = ensure_single_row_per_patient(clin_df, 'clin')

    blood_df = pd.read_csv(data_root / 'blood_mimm.csv')
    blood_df = blood_df.rename(columns=lambda x: f'blood_{x}' if x != 'patient' else x)
    blood_df = ensure_single_row_per_patient(blood_df, 'blood')

    radio_report_df = pd.read_csv(data_root / 'radioreports_mimm.csv')
    radio_report_df = radio_report_df.rename(columns=lambda x: f'radio_report_{x}' if x != 'patient' else x)
    radio_report_df = ensure_single_row_per_patient(radio_report_df, 'radio_report')

    dfs = {
        'path': path_df,
        'radio': radio_df,
        'clin': clin_df,
        'blood': blood_df,
        'radio_report': radio_report_df,
    }
    return labels_df, dfs, 'patient'


patients_df, dfs, patient_id_col = load_mimm_modalities_mean(DATA_ROOT)

common_patients = set(patients_df['patient'])
for name, df in dfs.items():
    common_patients &= set(df['patient'])
common_patients = sorted(common_patients)

patients_df = patients_df[patients_df['patient'].isin(common_patients)].copy()
patients_df = patients_df.set_index('patient').loc[common_patients].reset_index()

for name in list(dfs.keys()):
    dfs[name] = dfs[name][dfs[name]['patient'].isin(common_patients)].copy()
    dfs[name] = dfs[name].set_index('patient').loc[common_patients].reset_index()

modality_feature_dims = {name: int(df.shape[1] - 1) for name, df in dfs.items()}
print('Patients with all modalities and labels available:', len(common_patients))
print('Modality feature dimensions:', modality_feature_dims)


[radio_report] duplicated patients detected: 2. Collapsing by mean. Example IDs: [13656733, 16891439]
Patients with all modalities and labels available: 281
Modality feature dimensions: {'path': 768, 'radio': 4096, 'clin': 20, 'blood': 6, 'radio_report': 4096}


## 3. Feature Matrices

In [3]:
modalities_order = ['path', 'radio', 'clin', 'blood', 'radio_report']
modalities_order = [m for m in modalities_order if m in dfs]

X_modalities = {}
for modality_name in modalities_order:
    df = dfs[modality_name].copy()
    feature_cols = [c for c in df.columns if c != patient_id_col]
    X_modalities[modality_name] = df[feature_cols].to_numpy(dtype=np.float32, copy=True)

Y_classification = {target: patients_df[target].to_numpy(dtype=np.int64, copy=True) for target in CLASSIFICATION_TARGETS}
y_regression = patients_df[REGRESSION_TARGET].to_numpy(dtype=np.float32, copy=True)

print('Modalities used:', modalities_order)
for modality_name in modalities_order:
    print(modality_name, X_modalities[modality_name].shape)
print('Classification targets:', {k: Counter(v.tolist()) for k, v in Y_classification.items()}) if False else None
print('Regression target shape:', y_regression.shape)

Modalities used: ['path', 'radio', 'clin', 'blood', 'radio_report']
path (281, 768)
radio (281, 4096)
clin (281, 20)
blood (281, 6)
radio_report (281, 4096)
Regression target shape: (281,)


## 4. Helpers

In [4]:
from typing import Dict, List


def fit_pca_per_modality(X_train_modalities: Dict[str, np.ndarray], X_eval_modalities: Dict[str, np.ndarray], pca_variance: float):
    train_blocks = []
    eval_blocks = []
    component_info = []
    fitted = {}

    for modality_name in modalities_order:
        Xtr = np.asarray(X_train_modalities[modality_name], dtype=np.float32)
        Xev = np.asarray(X_eval_modalities[modality_name], dtype=np.float32)

        scaler = StandardScaler()
        Xtr_scaled = scaler.fit_transform(Xtr)
        Xev_scaled = scaler.transform(Xev)

        pca = PCA(n_components=pca_variance, svd_solver='full', random_state=RANDOM_STATE)
        Xtr_pca = pca.fit_transform(Xtr_scaled)
        Xev_pca = pca.transform(Xev_scaled)

        train_blocks.append(Xtr_pca)
        eval_blocks.append(Xev_pca)
        fitted[modality_name] = {'scaler': scaler, 'pca': pca}
        component_info.append({
            'modality': modality_name,
            'original_dim': Xtr.shape[1],
            'n_components': int(pca.n_components_),
            'explained_variance_ratio': float(np.sum(pca.explained_variance_ratio_)),
        })

    Xtr_concat = np.concatenate(train_blocks, axis=1)
    Xev_concat = np.concatenate(eval_blocks, axis=1)
    component_df = pd.DataFrame(component_info)
    return Xtr_concat, Xev_concat, fitted, component_df


def subset_modalities(X_modalities: Dict[str, np.ndarray], indices: np.ndarray):
    return {name: X_modalities[name][indices] for name in modalities_order}


def classification_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_prob_clip = np.clip(y_prob, 1e-7, 1 - 1e-7)
    y_pred = (y_prob >= threshold).astype(np.int64)

    if np.unique(y_true).size > 1:
        auc = float(roc_auc_score(y_true, y_prob))
        aucpr = float(average_precision_score(y_true, y_prob))
    else:
        auc = 0.5
        aucpr = float(y_true.mean())

    return {
        'auc': auc,
        'aucpr': aucpr,
        'accuracy': float((y_pred == y_true).mean()),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'logloss': float(log_loss(y_true, y_prob_clip, labels=[0, 1])),
    }


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if np.std(y_true) > 0 and np.std(y_pred) > 0 else 0.0
    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'pearson_r': corr,
    }


## 5. Classification Baseline: Nested CV With Logistic Regression

In [5]:
classification_outer_rows = []
classification_selected_rows = []
classification_component_rows = []

for target_name in CLASSIFICATION_TARGETS:
    print(f'\n=== Running classification target: {target_name} ===')
    y = Y_classification[target_name]
    outer_cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y), start=1):
        y_train = y[train_idx]
        y_test = y[test_idx]

        X_train_modalities = subset_modalities(X_modalities, train_idx)
        X_test_modalities = subset_modalities(X_modalities, test_idx)

        best_cfg = None
        best_inner_score = -np.inf
        best_inner_logloss = np.inf

        inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE + outer_fold)

        for pca_variance in PCA_VARIANCE_GRID:
            for C in LOGREG_C_GRID:
                for class_weight in LOGREG_CLASS_WEIGHT_GRID:
                    inner_scores = []
                    inner_loglosses = []

                    for inner_train_idx, inner_val_idx in inner_cv.split(np.zeros(len(y_train)), y_train):
                        X_inner_train_modalities = subset_modalities(X_train_modalities, inner_train_idx)
                        X_inner_val_modalities = subset_modalities(X_train_modalities, inner_val_idx)
                        y_inner_train = y_train[inner_train_idx]
                        y_inner_val = y_train[inner_val_idx]

                        X_inner_train, X_inner_val, _, _ = fit_pca_per_modality(
                            X_inner_train_modalities,
                            X_inner_val_modalities,
                            pca_variance=pca_variance,
                        )

                        clf = LogisticRegression(
                            C=C,
                            penalty='l2',
                            solver='lbfgs',
                            max_iter=5000,
                            class_weight=class_weight,
                            random_state=RANDOM_STATE,
                        )
                        clf.fit(X_inner_train, y_inner_train)
                        y_inner_prob = clf.predict_proba(X_inner_val)[:, 1]
                        inner_metrics = classification_metrics(y_inner_val, y_inner_prob)
                        inner_scores.append(inner_metrics['auc'])
                        inner_loglosses.append(inner_metrics['logloss'])

                    mean_inner_score = float(np.mean(inner_scores))
                    mean_inner_logloss = float(np.mean(inner_loglosses))

                    if (mean_inner_score > best_inner_score) or (np.isclose(mean_inner_score, best_inner_score) and mean_inner_logloss < best_inner_logloss):
                        best_inner_score = mean_inner_score
                        best_inner_logloss = mean_inner_logloss
                        best_cfg = {
                            'pca_variance': pca_variance,
                            'C': C,
                            'class_weight': class_weight,
                        }

        print(
            f"Target={target_name} | outer_fold={outer_fold} | selected_cfg={best_cfg} | inner_mean_auc={best_inner_score:.4f}"
        )

        X_train_concat, X_test_concat, _, component_df = fit_pca_per_modality(
            X_train_modalities,
            X_test_modalities,
            pca_variance=best_cfg['pca_variance'],
        )
        clf = LogisticRegression(
            C=best_cfg['C'],
            penalty='l2',
            solver='lbfgs',
            max_iter=5000,
            class_weight=best_cfg['class_weight'],
            random_state=RANDOM_STATE,
        )
        clf.fit(X_train_concat, y_train)
        y_test_prob = clf.predict_proba(X_test_concat)[:, 1]
        outer_metrics = classification_metrics(y_test, y_test_prob)

        classification_outer_rows.append({
            'target': target_name,
            'outer_fold': outer_fold,
            **best_cfg,
            'inner_selected_auc': best_inner_score,
            'inner_selected_logloss': best_inner_logloss,
            'n_train': int(len(train_idx)),
            'n_test': int(len(test_idx)),
            **outer_metrics,
        })

        classification_selected_rows.append({
            'target': target_name,
            'outer_fold': outer_fold,
            **best_cfg,
            'total_pca_components': int(X_train_concat.shape[1]),
        })

        component_df = component_df.copy()
        component_df['target'] = target_name
        component_df['outer_fold'] = outer_fold
        component_df['pca_variance'] = best_cfg['pca_variance']
        classification_component_rows.append(component_df)

classification_outer_df = pd.DataFrame(classification_outer_rows)
classification_selected_df = pd.DataFrame(classification_selected_rows)
classification_components_df = pd.concat(classification_component_rows, ignore_index=True)

classification_summary_df = (
    classification_outer_df
    .groupby('target', as_index=False)
    .agg(
        mean_outer_auc=('auc', 'mean'),
        std_outer_auc=('auc', 'std'),
        mean_outer_aucpr=('aucpr', 'mean'),
        std_outer_aucpr=('aucpr', 'std'),
        mean_outer_accuracy=('accuracy', 'mean'),
        std_outer_accuracy=('accuracy', 'std'),
        mean_outer_balanced_accuracy=('balanced_accuracy', 'mean'),
        std_outer_balanced_accuracy=('balanced_accuracy', 'std'),
        mean_outer_f1=('f1', 'mean'),
        std_outer_f1=('f1', 'std'),
        mean_outer_precision=('precision', 'mean'),
        std_outer_precision=('precision', 'std'),
        mean_outer_recall=('recall', 'mean'),
        std_outer_recall=('recall', 'std'),
        mean_outer_logloss=('logloss', 'mean'),
        std_outer_logloss=('logloss', 'std'),
    )
    .sort_values(['mean_outer_accuracy', 'mean_outer_auc'], ascending=False)
    .reset_index(drop=True)
)

print('=== Classification summary ===')
display(classification_summary_df)

best_classification_row = classification_summary_df.iloc[0]
print(
    f"Best classification label by mean outer accuracy: {best_classification_row['target']} "
    f"(accuracy={best_classification_row['mean_outer_accuracy']:.4f}, "
    f"AUC={best_classification_row['mean_outer_auc']:.4f})"
)



=== Running classification target: OS_3_label ===
Target=OS_3_label | outer_fold=1 | selected_cfg={'pca_variance': 0.9, 'C': 0.01, 'class_weight': None} | inner_mean_auc=0.6486
Target=OS_3_label | outer_fold=2 | selected_cfg={'pca_variance': 0.99, 'C': 0.1, 'class_weight': 'balanced'} | inner_mean_auc=0.5821
Target=OS_3_label | outer_fold=3 | selected_cfg={'pca_variance': 0.95, 'C': 0.01, 'class_weight': 'balanced'} | inner_mean_auc=0.5609
Target=OS_3_label | outer_fold=4 | selected_cfg={'pca_variance': 0.95, 'C': 0.01, 'class_weight': None} | inner_mean_auc=0.5992
Target=OS_3_label | outer_fold=5 | selected_cfg={'pca_variance': 0.9, 'C': 0.01, 'class_weight': None} | inner_mean_auc=0.7045

=== Running classification target: OS_4_label ===
Target=OS_4_label | outer_fold=1 | selected_cfg={'pca_variance': 0.9, 'C': 0.01, 'class_weight': None} | inner_mean_auc=0.6190
Target=OS_4_label | outer_fold=2 | selected_cfg={'pca_variance': 0.99, 'C': 0.01, 'class_weight': None} | inner_mean_auc=0

,target,mean_outer_auc,std_outer_auc,mean_outer_aucpr,std_outer_aucpr,mean_outer_accuracy,std_outer_accuracy,mean_outer_balanced_accuracy,std_outer_balanced_accuracy,mean_outer_f1,std_outer_f1,mean_outer_precision,std_outer_precision,mean_outer_recall,std_outer_recall,mean_outer_logloss,std_outer_logloss
0,OS_3_label,0.603732,0.079569,0.920679,0.019031,0.836404,0.038023,0.524939,0.093399,0.909307,0.022189,0.885561,0.019727,0.935592,0.045385,0.543438,0.119663
1,OS_4_label,0.581383,0.081692,0.868556,0.040280,0.750940,0.061555,0.543874,0.063373,0.850258,0.040325,0.833056,0.024549,0.869565,0.065217,0.935930,0.619123
2,OS_5_label,0.595470,0.138833,0.797537,0.078866,0.715163,0.061231,0.602300,0.102455,0.813747,0.035194,0.785966,0.054355,0.844599,0.014067,0.840625,0.238271
3,OS_9_label,0.594383,0.037255,0.629709,0.017590,0.590727,0.053668,0.589915,0.054413,0.612355,0.049489,0.609613,0.052016,0.620000,0.074896,1.794787,1.119655
4,OS_12_label,0.551504,0.095821,0.494313,0.077472,0.583584,0.062915,0.562171,0.075335,0.457178,0.125687,0.483854,0.092856,0.439493,0.154079,1.683565,1.546126
5,OS_6_label,0.569157,0.040917,0.690009,0.025790,0.576692,0.043693,0.550794,0.043753,0.656987,0.052905,0.676726,0.028744,0.642540,0.083732,1.075887,0.096227


Best classification label by mean outer accuracy: OS_3_label (accuracy=0.8364, AUC=0.6037)


## 6. Regression Baseline: Nested CV With Ridge

In [6]:
regression_outer_rows = []
regression_component_rows = []

outer_cv = KFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'\n=== Running regression target: {REGRESSION_TARGET} ===')
for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y_regression))), start=1):
    y_train = y_regression[train_idx]
    y_test = y_regression[test_idx]

    X_train_modalities = subset_modalities(X_modalities, train_idx)
    X_test_modalities = subset_modalities(X_modalities, test_idx)

    best_cfg = None
    best_inner_score = np.inf

    inner_cv = KFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE + outer_fold)

    for pca_variance in PCA_VARIANCE_GRID:
        for alpha in RIDGE_ALPHA_GRID:
            inner_rmses = []

            for inner_train_idx, inner_val_idx in inner_cv.split(np.zeros(len(y_train))):
                X_inner_train_modalities = subset_modalities(X_train_modalities, inner_train_idx)
                X_inner_val_modalities = subset_modalities(X_train_modalities, inner_val_idx)
                y_inner_train = y_train[inner_train_idx]
                y_inner_val = y_train[inner_val_idx]

                X_inner_train, X_inner_val, _, _ = fit_pca_per_modality(
                    X_inner_train_modalities,
                    X_inner_val_modalities,
                    pca_variance=pca_variance,
                )

                reg = Ridge(alpha=alpha, random_state=RANDOM_STATE)
                reg.fit(X_inner_train, y_inner_train)
                y_inner_pred = reg.predict(X_inner_val)
                inner_rmse = float(np.sqrt(mean_squared_error(y_inner_val, y_inner_pred)))
                inner_rmses.append(inner_rmse)

            mean_inner_rmse = float(np.mean(inner_rmses))
            if mean_inner_rmse < best_inner_score:
                best_inner_score = mean_inner_rmse
                best_cfg = {
                    'pca_variance': pca_variance,
                    'alpha': alpha,
                }

    print(
        f"Regression target={REGRESSION_TARGET} | outer_fold={outer_fold} | selected_cfg={best_cfg} | inner_mean_rmse={best_inner_score:.4f}"
    )

    X_train_concat, X_test_concat, _, component_df = fit_pca_per_modality(
        X_train_modalities,
        X_test_modalities,
        pca_variance=best_cfg['pca_variance'],
    )
    reg = Ridge(alpha=best_cfg['alpha'], random_state=RANDOM_STATE)
    reg.fit(X_train_concat, y_train)
    y_test_pred = reg.predict(X_test_concat)
    outer_metrics = regression_metrics(y_test, y_test_pred)

    regression_outer_rows.append({
        'target': REGRESSION_TARGET,
        'outer_fold': outer_fold,
        **best_cfg,
        'inner_selected_rmse': best_inner_score,
        'n_train': int(len(train_idx)),
        'n_test': int(len(test_idx)),
        **outer_metrics,
    })

    component_df = component_df.copy()
    component_df['target'] = REGRESSION_TARGET
    component_df['outer_fold'] = outer_fold
    component_df['pca_variance'] = best_cfg['pca_variance']
    regression_component_rows.append(component_df)

regression_outer_df = pd.DataFrame(regression_outer_rows)
regression_components_df = pd.concat(regression_component_rows, ignore_index=True)

regression_summary_df = (
    regression_outer_df
    .groupby('target', as_index=False)
    .agg(
        mean_outer_rmse=('rmse', 'mean'),
        std_outer_rmse=('rmse', 'std'),
        mean_outer_mae=('mae', 'mean'),
        std_outer_mae=('mae', 'std'),
        mean_outer_r2=('r2', 'mean'),
        std_outer_r2=('r2', 'std'),
        mean_outer_pearson_r=('pearson_r', 'mean'),
        std_outer_pearson_r=('pearson_r', 'std'),
    )
)

print('=== Regression summary ===')
display(regression_summary_df)



=== Running regression target: os_months ===


/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.60437e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.49638e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.37916e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.32958e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM

Regression target=os_months | outer_fold=1 | selected_cfg={'pca_variance': 0.99, 'alpha': 100.0} | inner_mean_rmse=21.6349


/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.53418e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.6109e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.43603e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.4865e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/l

Regression target=os_months | outer_fold=2 | selected_cfg={'pca_variance': 0.99, 'alpha': 100.0} | inner_mean_rmse=22.8321


...

/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.45903e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.50536e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.48015e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.31573e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM

Regression target=os_months | outer_fold=4 | selected_cfg={'pca_variance': 0.99, 'alpha': 100.0} | inner_mean_rmse=20.1659


/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.38254e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.62583e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.47782e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:237: LinAlgWarning: Ill-conditioned matrix (rcond=1.39114e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/TFM

Regression target=os_months | outer_fold=5 | selected_cfg={'pca_variance': 0.99, 'alpha': 100.0} | inner_mean_rmse=21.7028
=== Regression summary ===


,target,mean_outer_rmse,std_outer_rmse,mean_outer_mae,std_outer_mae,mean_outer_r2,std_outer_r2,mean_outer_pearson_r,std_outer_pearson_r
0,os_months,24.610921,1.825495,18.172909,1.334726,-1.253347,1.198785,-0.083858,0.126679


## 7. Selected Configurations And PCA Footprint

In [7]:
print('=== Selected classification configs per outer fold ===')
display(classification_selected_df.sort_values(['target', 'outer_fold']).reset_index(drop=True))

classification_pca_summary_df = (
    classification_components_df
    .groupby(['target', 'modality'], as_index=False)
    .agg(
        mean_n_components=('n_components', 'mean'),
        min_n_components=('n_components', 'min'),
        max_n_components=('n_components', 'max'),
        mean_explained_variance_ratio=('explained_variance_ratio', 'mean'),
    )
    .sort_values(['target', 'modality'])
    .reset_index(drop=True)
)

print('=== Classification PCA summary ===')
display(classification_pca_summary_df)

regression_pca_summary_df = (
    regression_components_df
    .groupby(['target', 'modality'], as_index=False)
    .agg(
        mean_n_components=('n_components', 'mean'),
        min_n_components=('n_components', 'min'),
        max_n_components=('n_components', 'max'),
        mean_explained_variance_ratio=('explained_variance_ratio', 'mean'),
    )
)

print('=== Regression PCA summary ===')
display(regression_pca_summary_df)

=== Selected classification configs per outer fold ===


,target,outer_fold,pca_variance,C,class_weight,total_pca_components
0,OS_12_label,1,0.99,0.01,None,506
1,OS_12_label,2,0.99,0.01,None,503
2,OS_12_label,3,0.99,0.01,None,506
3,OS_12_label,4,0.90,10.00,balanced,214
4,OS_12_label,5,0.95,0.01,None,310
5,OS_3_label,1,0.90,0.01,None,212
6,OS_3_label,2,0.99,0.10,balanced,505
7,OS_3_label,3,0.95,0.01,balanced,312
8,OS_3_label,4,0.95,0.01,None,311
9,OS_3_label,5,0.90,0.01,None,212


=== Classification PCA summary ===


,target,modality,mean_n_components,min_n_components,max_n_components,mean_explained_variance_ratio
0,OS_12_label,blood,5.8,5,6,0.985636
1,OS_12_label,clin,16.0,14,17,0.974903
2,OS_12_label,path,96.6,38,129,0.964923
3,OS_12_label,radio,112.4,38,151,0.964379
4,OS_12_label,radio_report,177.0,119,204,0.964341
5,OS_3_label,blood,5.6,5,6,0.969767
6,OS_3_label,clin,15.0,14,17,0.952783
7,OS_3_label,path,65.0,37,127,0.939034
8,OS_3_label,radio,74.4,37,151,0.938447
9,OS_3_label,radio_report,150.4,118,204,0.938517


=== Regression PCA summary ===


,target,modality,mean_n_components,min_n_components,max_n_components,mean_explained_variance_ratio
0,os_months,blood,6.0,6,6,1.000000
1,os_months,clin,17.0,17,17,0.999144
2,os_months,path,128.0,127,129,0.990175
3,os_months,radio,150.6,150,151,0.990167
4,os_months,radio_report,203.8,203,204,0.990260


## 8. Final Conclusion

In [8]:
print('=== Final classification ranking by mean outer accuracy ===')
display(classification_summary_df[['target', 'mean_outer_accuracy', 'mean_outer_auc', 'mean_outer_aucpr', 'mean_outer_balanced_accuracy', 'mean_outer_f1', 'mean_outer_logloss']])

print('=== Final regression performance ===')
display(regression_summary_df)

=== Final classification ranking by mean outer accuracy ===


,target,mean_outer_accuracy,mean_outer_auc,mean_outer_aucpr,mean_outer_balanced_accuracy,mean_outer_f1,mean_outer_logloss
0,OS_3_label,0.836404,0.603732,0.920679,0.524939,0.909307,0.543438
1,OS_4_label,0.750940,0.581383,0.868556,0.543874,0.850258,0.935930
2,OS_5_label,0.715163,0.595470,0.797537,0.602300,0.813747,0.840625
3,OS_9_label,0.590727,0.594383,0.629709,0.589915,0.612355,1.794787
4,OS_12_label,0.583584,0.551504,0.494313,0.562171,0.457178,1.683565
5,OS_6_label,0.576692,0.569157,0.690009,0.550794,0.656987,1.075887


=== Final regression performance ===


,target,mean_outer_rmse,std_outer_rmse,mean_outer_mae,std_outer_mae,mean_outer_r2,std_outer_r2,mean_outer_pearson_r,std_outer_pearson_r
0,os_months,24.610921,1.825495,18.172909,1.334726,-1.253347,1.198785,-0.083858,0.126679
